In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FOLDER = Path(
    r"C:\Users\admin\Anaconda Project\Project 3 New\MedCore_RCM_Data\MedCore_RCM_Data_Corrected"
)

print("Folder exists:", DATA_FOLDER.exists())
print("CSV files found:")

for file in sorted(DATA_FOLDER.glob("*.csv")):
    print(file.name)

Folder exists: True
CSV files found:
Authorization.csv
Claims.csv
Patients.csv
Payers.csv
Payments.csv
Procedures.csv
Providers.csv


In [2]:
csv_files = {
    "patients": "Patients.csv",
    "payers": "Payers.csv",
    "providers": "Providers.csv",
    "procedures": "Procedures.csv",
    "claims": "Claims.csv",
    "authorization": "Authorization.csv",
    "payments": "Payments.csv"
}

data = {
    name: pd.read_csv(DATA_FOLDER / filename)
    for name, filename in csv_files.items()
}

for name, df in data.items():
    print(f"{name.title()}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Patients: 15,000 rows, 3 columns
Payers: 8 rows, 3 columns
Providers: 40 rows, 5 columns
Procedures: 24 rows, 6 columns
Claims: 50,000 rows, 12 columns
Authorization: 50,000 rows, 6 columns
Payments: 50,000 rows, 5 columns


In [3]:
for name, df in data.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {name.upper()}")
    print("=" * 50)

    print("\nColumn data types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nDuplicate rows:", df.duplicated().sum())


TABLE: PATIENTS

Column data types:
Patient_ID    object
Age            int64
Gender        object
dtype: object

Missing values:
Patient_ID    0
Age           0
Gender        0
dtype: int64

Duplicate rows: 0

TABLE: PAYERS

Column data types:
Payer_ID      object
Payer_Name    object
Payer_Type    object
dtype: object

Missing values:
Payer_ID      0
Payer_Name    0
Payer_Type    0
dtype: int64

Duplicate rows: 0

TABLE: PROVIDERS

Column data types:
Provider_ID          object
Provider_Name        object
Location             object
Provider_Type        object
Specialty_Profile    object
dtype: object

Missing values:
Provider_ID          0
Provider_Name        0
Location             0
Provider_Type        0
Specialty_Profile    0
dtype: int64

Duplicate rows: 0

TABLE: PROCEDURES

Column data types:
Procedure_Code            object
Procedure_Name            object
Specialty                 object
Authorization_Required      bool
Min_Claim_Amount           int64
Max_Claim_Amount    

In [4]:
claims = data["claims"]
authorization = data["authorization"]
payments = data["payments"]

print("1. DENIAL REASON CHECK")
print(claims.groupby("Claim_Status")["Denial_Reason"].apply(
    lambda s: s.isna().sum()
))

print("\n2. AUTHORIZATION STATUS CHECK")
print(authorization.groupby("Authorization_Status").agg(
    Total_Claims=("Claim_ID", "count"),
    Missing_Request_Date=("Request_Date", lambda s: s.isna().sum()),
    Missing_Approval_Date=("Approval_Date", lambda s: s.isna().sum()),
    Missing_Authorization_ID=("Authorization_ID", lambda s: s.isna().sum())
))

print("\n3. PAYMENT DATE CHECK")
print(payments.groupby("Recovery_Status").agg(
    Total_Claims=("Claim_ID", "count"),
    Zero_Paid_Amount=("Paid_Amount", lambda s: (s == 0).sum()),
    Missing_Payment_Date=("Payment_Date", lambda s: s.isna().sum())
))

1. DENIAL REASON CHECK
Claim_Status
Denied         0
Paid       43113
Pending     2750
Name: Denial_Reason, dtype: int64

2. AUTHORIZATION STATUS CHECK
                      Total_Claims  Missing_Request_Date  \
Authorization_Status                                       
Approved                     12642                     0   
Denied                         871                     0   
Not Requested                 2002                  2002   
Not Required                 33340                 33340   
Pending                       1145                     0   

                      Missing_Approval_Date  Missing_Authorization_ID  
Authorization_Status                                                   
Approved                                  0                         0  
Denied                                  871                         0  
Not Requested                          2002                      2002  
Not Required                          33340                     333

In [5]:
claims = data["claims"]
authorization = data["authorization"]
procedures = data["procedures"]

# Check authorization-required flag against procedure master data
check = (
    claims[["Claim_ID", "Procedure_Code"]]
    .merge(
        procedures[["Procedure_Code", "Authorization_Required"]],
        on="Procedure_Code",
        how="left"
    )
    .merge(
        authorization[["Claim_ID", "Authorization_Required",
                       "Authorization_Status"]],
        on="Claim_ID",
        how="left",
        suffixes=("_procedure", "_authorization")
    )
)

print("Claims checked:", len(check))

print("\nProcedure flag vs authorization flag:")
print(
    (check["Authorization_Required_procedure"]
     != check["Authorization_Required_authorization"]).sum()
)

print("\nAuthorization status vs required flag:")
print(pd.crosstab(
    check["Authorization_Required_procedure"],
    check["Authorization_Status"],
    margins=True
))

Claims checked: 50000

Procedure flag vs authorization flag:
0

Authorization status vs required flag:
Authorization_Status              Approved  Denied  Not Requested  \
Authorization_Required_procedure                                    
False                                    0       0              0   
True                                 12642     871           2002   
All                                  12642     871           2002   

Authorization_Status              Not Required  Pending    All  
Authorization_Required_procedure                                
False                                    33340        0  33340  
True                                         0     1145  16660  
All                                      33340     1145  50000  


In [6]:
claims = data["claims"]
procedures = data["procedures"]

amount_check = claims.merge(
    procedures[[
        "Procedure_Code",
        "Min_Claim_Amount",
        "Max_Claim_Amount"
    ]],
    on="Procedure_Code",
    how="left"
)

below_minimum = (
    amount_check["Claim_Amount"]
    < amount_check["Min_Claim_Amount"]
)

above_maximum = (
    amount_check["Claim_Amount"]
    > amount_check["Max_Claim_Amount"]
)

print("Total claims checked:", len(amount_check))
print("Claims below minimum:", below_minimum.sum())
print("Claims above maximum:", above_maximum.sum())
print("Claims within range:", (~(below_minimum | above_maximum)).sum())

Total claims checked: 50000
Claims below minimum: 0
Claims above maximum: 0
Claims within range: 50000


In [7]:
claims = data["claims"]

print("Claim status counts:")
print(claims["Claim_Status"].value_counts())

print("\nDenial flag counts:")
print(claims["Denial_Flag"].value_counts())

print("\nDenied claims with an incorrect flag:",
      ((claims["Claim_Status"] == "Denied") &
       (claims["Denial_Flag"] != 1)).sum())

print("\nNon-denied claims with an incorrect flag:",
      ((claims["Claim_Status"] != "Denied") &
       (claims["Denial_Flag"] != 0)).sum())

Claim status counts:
Claim_Status
Paid       43113
Denied      4137
Pending     2750
Name: count, dtype: int64

Denial flag counts:
Denial_Flag
0    45863
1     4137
Name: count, dtype: int64

Denied claims with an incorrect flag: 0

Non-denied claims with an incorrect flag: 0


In [8]:
claims = data["claims"]
payments = data["payments"]

payment_check = claims[
    ["Claim_ID", "Claim_Amount"]
].merge(
    payments[
        ["Claim_ID", "Paid_Amount", "Outstanding_Amount"]
    ],
    on="Claim_ID",
    how="left"
)

negative_payments = payment_check["Paid_Amount"] < 0

overpayments = (
    payment_check["Paid_Amount"] >
    payment_check["Claim_Amount"]
)

incorrect_outstanding = ~np.isclose(
    payment_check["Outstanding_Amount"],
    payment_check["Claim_Amount"] - payment_check["Paid_Amount"],
    atol=0.01
)

print("Total claims checked:", len(payment_check))
print("Negative payments:", negative_payments.sum())
print("Payments exceeding claim amount:", overpayments.sum())
print("Incorrect outstanding amounts:", incorrect_outstanding.sum())

Total claims checked: 50000
Negative payments: 0
Payments exceeding claim amount: 0
Incorrect outstanding amounts: 0


In [9]:
claims = data["claims"]
payments = data["payments"]

status_check = claims[
    ["Claim_ID", "Claim_Status"]
].merge(
    payments[["Claim_ID", "Recovery_Status"]],
    on="Claim_ID",
    how="left"
)

print("Claim status vs recovery status:")
print(pd.crosstab(
    status_check["Claim_Status"],
    status_check["Recovery_Status"],
    margins=True
))

print("\nPaid claims not marked Collected:",
      ((status_check["Claim_Status"] == "Paid") &
       (status_check["Recovery_Status"] != "Collected")).sum())

print("\nPending claims not marked Pending:",
      ((status_check["Claim_Status"] == "Pending") &
       (status_check["Recovery_Status"] != "Pending")).sum())

Claim status vs recovery status:
Recovery_Status  Collected  Pending  Recovered  Unrecovered    All
Claim_Status                                                      
Denied                   0        0       2513         1624   4137
Paid                 43113        0          0            0  43113
Pending                  0     2750          0            0   2750
All                  43113     2750       2513         1624  50000

Paid claims not marked Collected: 0

Pending claims not marked Pending: 0


In [10]:
claims = data["claims"]
authorization = data["authorization"]
payments = data["payments"]

# Convert dates in temporary Series for checking only
service_dates = pd.to_datetime(
    claims["Service_Date"], errors="coerce"
)
request_dates = pd.to_datetime(
    authorization["Request_Date"], errors="coerce"
)
approval_dates = pd.to_datetime(
    authorization["Approval_Date"], errors="coerce"
)
payment_dates = pd.to_datetime(
    payments["Payment_Date"], errors="coerce"
)

print("Invalid service dates:", service_dates.isna().sum())
print("Service dates outside 2025:",
      (~service_dates.between("2025-01-01", "2025-12-31")).sum())

print("\nInvalid non-missing request dates:",
      (authorization["Request_Date"].notna() & request_dates.isna()).sum())
print("Invalid non-missing approval dates:",
      (authorization["Approval_Date"].notna() & approval_dates.isna()).sum())
print("Invalid non-missing payment dates:",
      (payments["Payment_Date"].notna() & payment_dates.isna()).sum())

print("\nApproval dates before request dates:",
      (approval_dates.notna() & request_dates.notna() &
       (approval_dates < request_dates)).sum())

Invalid service dates: 0
Service dates outside 2025: 0

Invalid non-missing request dates: 0
Invalid non-missing approval dates: 0
Invalid non-missing payment dates: 0

Approval dates before request dates: 0


In [11]:
claims = data["claims"]
authorization = data["authorization"]

auth_timing = claims[
    ["Claim_ID", "Service_Date"]
].merge(
    authorization[
        ["Claim_ID", "Authorization_Required",
         "Authorization_Status", "Approval_Date"]
    ],
    on="Claim_ID",
    how="left"
)

auth_timing["Service_Date"] = pd.to_datetime(
    auth_timing["Service_Date"], errors="coerce"
)

auth_timing["Approval_Date"] = pd.to_datetime(
    auth_timing["Approval_Date"], errors="coerce"
)

approved = auth_timing[
    auth_timing["Authorization_Status"] == "Approved"
]

approved_after_service = approved[
    approved["Approval_Date"] > approved["Service_Date"]
]

print("Approved claims:", len(approved))
print("Approved after service date:", len(approved_after_service))
print("Approved on or before service date:",
      (approved["Approval_Date"] <= approved["Service_Date"]).sum())

print("\nAuthorization statuses for required procedures:")
print(
    auth_timing[
        auth_timing["Authorization_Required"]
    ]["Authorization_Status"].value_counts()
)

Approved claims: 12642
Approved after service date: 2809
Approved on or before service date: 9833

Authorization statuses for required procedures:
Authorization_Status
Approved         12642
Not Requested     2002
Pending           1145
Denied             871
Name: count, dtype: int64


In [12]:
claims = data["claims"]
authorization = data["authorization"]

timing_analysis = claims[
    ["Claim_ID", "Claim_Status", "Service_Date"]
].merge(
    authorization[
        ["Claim_ID", "Authorization_Status", "Approval_Date"]
    ],
    on="Claim_ID",
    how="left"
)

timing_analysis["Service_Date"] = pd.to_datetime(
    timing_analysis["Service_Date"]
)
timing_analysis["Approval_Date"] = pd.to_datetime(
    timing_analysis["Approval_Date"],
    errors="coerce"
)

timing_analysis["Approval_Timing"] = "Not Approved"

approved_mask = (
    timing_analysis["Authorization_Status"] == "Approved"
)

timing_analysis.loc[
    approved_mask &
    (timing_analysis["Approval_Date"] <= timing_analysis["Service_Date"]),
    "Approval_Timing"
] = "On or Before Service"

timing_analysis.loc[
    approved_mask &
    (timing_analysis["Approval_Date"] > timing_analysis["Service_Date"]),
    "Approval_Timing"
] = "After Service"

print(pd.crosstab(
    timing_analysis["Approval_Timing"],
    timing_analysis["Claim_Status"],
    margins=True
))

Claim_Status          Denied   Paid  Pending    All
Approval_Timing                                    
After Service            337   2335      137   2809
Not Approved            2727  32571     2060  37358
On or Before Service    1073   8207      553   9833
All                     4137  43113     2750  50000


In [13]:
claims = data["claims"]
authorization = data["authorization"]

denial_analysis = claims[
    ["Claim_ID", "Claim_Status", "Denial_Flag"]
].merge(
    authorization[
        ["Claim_ID", "Authorization_Status"]
    ],
    on="Claim_ID",
    how="left"
)

status_summary = denial_analysis.groupby(
    "Authorization_Status"
).agg(
    Total_Claims=("Claim_ID", "count"),
    Denied_Claims=("Denial_Flag", "sum")
)

status_summary["Denial_Rate_Pct"] = (
    status_summary["Denied_Claims"]
    / status_summary["Total_Claims"] * 100
).round(2)

print(status_summary.sort_values(
    "Denial_Rate_Pct",
    ascending=False
))

                      Total_Claims  Denied_Claims  Denial_Rate_Pct
Authorization_Status                                              
Pending                       1145            137            11.97
Approved                     12642           1410            11.15
Denied                         871             92            10.56
Not Requested                 2002            192             9.59
Not Required                 33340           2306             6.92


In [14]:
claims = data["claims"]

denied_claims = claims[
    claims["Claim_Status"] == "Denied"
]

denial_reason_summary = (
    denied_claims["Denial_Reason"]
    .value_counts()
    .rename_axis("Denial_Reason")
    .reset_index(name="Denied_Claims")
)

denial_reason_summary["Percentage"] = (
    denial_reason_summary["Denied_Claims"]
    / len(denied_claims) * 100
).round(2)

print("Total denied claims:", len(denied_claims))
print(denial_reason_summary.to_string(index=False))

Total denied claims: 4137
        Denial_Reason  Denied_Claims  Percentage
         Coding Error            946       22.87
Documentation Missing            916       22.14
    Medical Necessity            822       19.87
    Eligibility Issue            775       18.73
        Timely Filing            265        6.41
Missing Authorization            236        5.70
      Duplicate Claim            177        4.28


In [15]:
claims = data["claims"]

denial_amount_summary = (
    claims[claims["Claim_Status"] == "Denied"]
    .groupby("Denial_Reason")
    .agg(
        Denied_Claims=("Claim_ID", "count"),
        Total_Denied_Amount=("Claim_Amount", "sum"),
        Average_Claim_Amount=("Claim_Amount", "mean"),
        Median_Claim_Amount=("Claim_Amount", "median")
    )
    .sort_values("Total_Denied_Amount", ascending=False)
)

denial_amount_summary["Total_Denied_Amount"] = (
    denial_amount_summary["Total_Denied_Amount"].round(2)
)

denial_amount_summary["Average_Claim_Amount"] = (
    denial_amount_summary["Average_Claim_Amount"].round(2)
)

denial_amount_summary["Median_Claim_Amount"] = (
    denial_amount_summary["Median_Claim_Amount"].round(2)
)

print(denial_amount_summary.to_string())

                       Denied_Claims  Total_Denied_Amount  Average_Claim_Amount  Median_Claim_Amount
Denial_Reason                                                                                       
Coding Error                     946          21101344.42              22305.86              4528.26
Medical Necessity                822          18669104.68              22711.81              5429.33
Eligibility Issue                775          17651598.51              22776.26              4427.07
Documentation Missing            916          14350297.58              15666.26              3666.83
Missing Authorization            236          10585738.83              44854.83             13659.21
Timely Filing                    265            868470.67               3277.25              2080.26
Duplicate Claim                  177            493342.75               2787.25              2087.08


In [16]:
claims = data["claims"]

total_claim_amount = claims["Claim_Amount"].sum()

denied = claims[claims["Claim_Status"] == "Denied"]
total_denied_amount = denied["Claim_Amount"].sum()

print(f"Total claim amount: ₹{total_claim_amount:,.2f}")
print(f"Denied claim amount: ₹{total_denied_amount:,.2f}")
print(
    "Denied amount as % of total claim amount: "
    f"{total_denied_amount / total_claim_amount * 100:.2f}%"
)

Total claim amount: ₹781,292,204.12
Denied claim amount: ₹83,719,897.44
Denied amount as % of total claim amount: 10.72%


In [17]:
claims = data["claims"]
authorization = data["authorization"]
payments = data["payments"]

print("Unique claim IDs:", claims["Claim_ID"].nunique())
print("Unique authorization claim IDs:", authorization["Claim_ID"].nunique())
print("Unique payment claim IDs:", payments["Claim_ID"].nunique())

print(
    "\nClaims missing authorization records:",
    len(set(claims["Claim_ID"]) - set(authorization["Claim_ID"]))
)

print(
    "Claims missing payment records:",
    len(set(claims["Claim_ID"]) - set(payments["Claim_ID"]))
)

print(
    "Authorization records without matching claims:",
    len(set(authorization["Claim_ID"]) - set(claims["Claim_ID"]))
)

print(
    "Payment records without matching claims:",
    len(set(payments["Claim_ID"]) - set(claims["Claim_ID"]))
)

Unique claim IDs: 50000
Unique authorization claim IDs: 50000
Unique payment claim IDs: 50000

Claims missing authorization records: 0
Claims missing payment records: 0
Authorization records without matching claims: 0
Payment records without matching claims: 0


In [20]:
print("Claims columns:")
print(data["claims"].columns.tolist())

print("\nProcedures columns:")
print(data["procedures"].columns.tolist())

Claims columns:
['Claim_ID', 'Patient_ID', 'Provider_ID', 'Payer_ID', 'Service_Date', 'Specialty', 'Procedure_Code', 'Procedure_Name', 'Claim_Amount', 'Denial_Flag', 'Denial_Reason', 'Claim_Status']

Procedures columns:
['Procedure_Code', 'Procedure_Name', 'Specialty', 'Authorization_Required', 'Min_Claim_Amount', 'Max_Claim_Amount']


In [21]:
claims = data["claims"]
patients = data["patients"]
payers = data["payers"]
providers = data["providers"]
procedures = data["procedures"]

checks = {
    "Patient_ID": patients["Patient_ID"],
    "Payer_ID": payers["Payer_ID"],
    "Provider_ID": providers["Provider_ID"],
    "Procedure_Code": procedures["Procedure_Code"]
}

for column, master_ids in checks.items():
    invalid_ids = ~claims[column].isin(master_ids)

    print(f"{column} - Invalid references: {invalid_ids.sum()}")

Patient_ID - Invalid references: 0
Payer_ID - Invalid references: 0
Provider_ID - Invalid references: 0
Procedure_Code - Invalid references: 0


In [22]:
claims = data["claims"]
procedures = data["procedures"]

procedure_check = claims.merge(
    procedures[["Procedure_Code", "Procedure_Name", "Specialty"]],
    on="Procedure_Code",
    how="left",
    suffixes=("_claim", "_master")
)

print(
    "Procedure name mismatches:",
    (
        procedure_check["Procedure_Name_claim"]
        != procedure_check["Procedure_Name_master"]
    ).sum()
)

print(
    "Specialty mismatches:",
    (
        procedure_check["Specialty_claim"]
        != procedure_check["Specialty_master"]
    ).sum()
)

Procedure name mismatches: 0
Specialty mismatches: 0


In [23]:
master_tables = {
    "Patients": (data["patients"], "Patient_ID"),
    "Payers": (data["payers"], "Payer_ID"),
    "Providers": (data["providers"], "Provider_ID"),
    "Procedures": (data["procedures"], "Procedure_Code")
}

for table_name, (df, key_column) in master_tables.items():
    duplicate_count = df[key_column].duplicated().sum()

    print(f"{table_name} - Duplicate {key_column}: {duplicate_count}")

Patients - Duplicate Patient_ID: 0
Payers - Duplicate Payer_ID: 0
Providers - Duplicate Provider_ID: 0
Procedures - Duplicate Procedure_Code: 0


In [24]:
transaction_tables = {
    "Claims": (data["claims"], "Claim_ID"),
    "Authorization": (data["authorization"], "Claim_ID"),
    "Payments": (data["payments"], "Claim_ID")
}

for table_name, (df, key_column) in transaction_tables.items():
    duplicate_count = df[key_column].duplicated().sum()

    print(f"{table_name} - Duplicate {key_column}: {duplicate_count}")

Claims - Duplicate Claim_ID: 0
Authorization - Duplicate Claim_ID: 0
Payments - Duplicate Claim_ID: 0


In [25]:
for table_name, df in data.items():
    print(
        f"{table_name.title():15} | "
        f"Rows: {df.shape[0]:,} | "
        f"Columns: {df.shape[1]}"
    )

Patients        | Rows: 15,000 | Columns: 3
Payers          | Rows: 8 | Columns: 3
Providers       | Rows: 40 | Columns: 5
Procedures      | Rows: 24 | Columns: 6
Claims          | Rows: 50,000 | Columns: 12
Authorization   | Rows: 50,000 | Columns: 6
Payments        | Rows: 50,000 | Columns: 5


In [ ]:
# MedCore RCM — Healthcare Claim Denial Analytics
## Data Profiling and Quality Validation

### 1. Project Overview
This project uses a simulated healthcare revenue-cycle dataset to analyze insurance claims, claim denials, authorization status, and payment recovery patterns.

### 2. Dataset Structure

| Table | Rows | Columns |
|---|---:|---:|
| Patients | 15,000 | 3 |
| Payers | 8 | 3 |
| Providers | 40 | 5 |
| Procedures | 24 | 6 |
| Claims | 50,000 | 12 |
| Authorization | 50,000 | 6 |
| Payments | 50,000 | 5 |

### 3. Data Quality Checks Performed
- Checked table dimensions and duplicate rows.
- Reviewed missing values and assessed whether they were consistent with record statuses.
- Validated claim status against the denial flag.
- Checked authorization requirements against the procedure master table.
- Validated claim amounts against procedure-level minimum and maximum amounts.
- Checked payment amounts and outstanding balances.
- Validated date fields and authorization timing.
- Verified claim-to-authorization and claim-to-payment relationships.
- Checked patient, payer, provider, and procedure references.
- Verified procedure names and specialties against the Procedures master table.
- Checked uniqueness of master-table IDs and transaction-table Claim_ID values.

### 4. Key Findings
- The Claims table contains 50,000 records.
- The claim-based denial rate is 8.27% (4,137 denied claims out of 50,000).
- Coding Error and Documentation Missing account for approximately 45.01% of denied claims.
- The original claim amount associated with denied claims is approximately ₹8.37 crore. This represents denied claim value, not confirmed financial loss.
- All tested master-table references were valid.
- No duplicate key values were found in the checked master and transaction tables.

### 5. Conclusion
The initial profiling and key relationship checks found no invalid references or duplicate key values in the tested tables. Missing values were also reviewed in the context of claim and authorization statuses.

Further analysis will focus on denial patterns, authorization outcomes, payment recovery, and financial exposure using SQL and Power BI.